In [7]:
import torch
import torch.nn as nn

!pip install onnx onnxruntime

class CNN1D(nn.Module):
    def __init__(self, num_classes=4):
        super(CNN1D, self).__init__()

        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)

        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.5)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)

        return x



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN1D(num_classes=4).to(device)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/nsl_kdd/cnn_model_final.pth",
        map_location=device
    )
)

model.eval()

print("Model loaded.")

dummy_input = torch.randn(1, 1, 122).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "/content/drive/MyDrive/nsl_kdd/cnn_model_phase7_single.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    opset_version=18,
    do_constant_folding=True,
    dynamo=False
)

print("Model exported to ONNX.")

import onnxruntime as ort
import numpy as np

onnx_path = "/content/drive/MyDrive/nsl_kdd/cnn_model_phase7_single.onnx"

ort_session = ort.InferenceSession(onnx_path)

print("ONNX model loaded.")


# Prepare sample input
sample = np.random.randn(1, 1, 122).astype(np.float32)

outputs = ort_session.run(
    None,
    {"input": sample}
)

print("ONNX output shape:", outputs[0].shape)


base_path = "/content/drive/MyDrive/nsl_kdd/"
X_test = np.load(base_path + "X_test.npy")

# Add channel dimension
X_test_onnx = X_test[:, np.newaxis, :].astype(np.float32)


import time

batch_size = 256

start_time = time.time()

for i in range(0, len(X_test_onnx), batch_size):
    batch = X_test_onnx[i:i+batch_size]
    ort_session.run(None, {"input": batch})

end_time = time.time()

total_time = end_time - start_time
num_samples = len(X_test_onnx)

time_per_sample = total_time / num_samples
throughput = num_samples / total_time

print(f"ONNX Total inference time: {total_time:.4f} sec")
print(f"ONNX Time per sample: {time_per_sample*1000:.6f} ms")
print(f"ONNX Throughput: {throughput:.2f} samples/sec")

import pandas as pd

onnx_results = pd.DataFrame({
    "Model": ["ONNX Runtime"],
    "Total_Time_sec": [total_time],
    "Time_per_Sample_ms": [time_per_sample * 1000],
    "Throughput_samples_per_sec": [throughput],
    "Batch_Size": [batch_size]
})

onnx_results.to_csv(
    "/content/drive/MyDrive/nsl_kdd/onnx_latency_results_phase7.csv",
    index=False
)

print("ONNX latency results saved.")


# ================================
# PyTorch CPU Latency Measurement
# ================================

import time

model.eval()

start_time_pt = time.time()

with torch.no_grad():
    for i in range(0, len(X_test), batch_size):
        batch = torch.tensor(
            X_test[i:i+batch_size],
            dtype=torch.float32
        ).unsqueeze(1)  # stay on CPU (DO NOT .to(device))

        outputs = model(batch)

end_time_pt = time.time()

total_time_pt = end_time_pt - start_time_pt
time_per_sample_pt = total_time_pt / len(X_test)
throughput_pt = len(X_test) / total_time_pt

print(f"PyTorch CPU Total inference time: {total_time_pt:.4f} sec")
print(f"PyTorch CPU Time per sample: {time_per_sample_pt*1000:.6f} ms")
print(f"PyTorch CPU Throughput: {throughput_pt:.2f} samples/sec")


# ================================
# Prediction Equivalence Validation
# ================================

# Get PyTorch predictions
model.eval()
pytorch_preds = []

with torch.no_grad():
    for i in range(0, len(X_test), batch_size):
        batch = torch.tensor(
            X_test[i:i+batch_size],
            dtype=torch.float32
        ).unsqueeze(1).to(device)

        outputs = model(batch)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        pytorch_preds.extend(preds)

pytorch_preds = np.array(pytorch_preds)


# Get ONNX predictions
onnx_preds = []

for i in range(0, len(X_test_onnx), batch_size):
    batch = X_test_onnx[i:i+batch_size]
    outputs = ort_session.run(None, {"input": batch})
    preds = np.argmax(outputs[0], axis=1)
    onnx_preds.extend(preds)

onnx_preds = np.array(onnx_preds)


# Compare
match_percentage = np.mean(pytorch_preds == onnx_preds) * 100

print(f"Prediction Match Percentage: {match_percentage:.4f}%")


Model loaded.
Model exported to ONNX.
ONNX model loaded.
ONNX output shape: (1, 4)


/tmp/ipython-input-2586825759.py:59: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX Total inference time: 1.4770 sec
ONNX Time per sample: 0.065515 ms
ONNX Throughput: 15263.75 samples/sec
ONNX latency results saved.
PyTorch CPU Total inference time: 2.0119 sec
PyTorch CPU Time per sample: 0.089243 ms
PyTorch CPU Throughput: 11205.41 samples/sec
Prediction Match Percentage: 100.0000%


Mounted at /content/drive


In [8]:
import os

base_path = "/content/drive/MyDrive/nsl_kdd/"

for root, dirs, files in os.walk(base_path):
    print(f"\n📂 {root}")
    for file in files:
        print("   ├──", file)



📂 /content/drive/MyDrive/nsl_kdd/
   ├── KDDTest+.arff
   ├── KDDTrain+.arff
   ├── KDDTest+.txt
   ├── KDDTrain+.txt
   ├── nsl_kdd_train_processed.csv
   ├── X_train.npy
   ├── X_test.npy
   ├── explainability_times_phase6.csv
   ├── cnn_model_phase7.onnx.data
   ├── cnn_model_phase7.onnx
   ├── cnn_model_phase7_single.onnx
   ├── onnx_latency_results_phase7.csv
   ├── nsl_kdd_test_processed.csv
   ├── preprocessor.pkl
   ├── label_encoder.pkl
   ├── y_test.npy
   ├── y_train.npy
   ├── cnn_final_results.csv
   ├── cnn_model_final.pth
   ├── cnn_classification_report.csv
   ├── cnn_latency_batch256_phase4.csv
   ├── autoencoder_results_phase5.csv
   ├── autoencoder_model_phase5.pth
   ├── shap_global_importance_phase6.csv
   ├── shap_stability_metrics_phase6.csv
   ├── explainability_latency_phase6.csv

📂 /content/drive/MyDrive/nsl_kdd/.ipynb_checkpoints


In [ ]:
!pip install onnx onnxruntime onnxscript


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.3/159.3 kB 10.9 MB/s eta 0:00:00
